In [1]:
from cohirf.experiment.open_ml_clustering_experiment import OpenmlClusteringExperiment
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import rgb2hex
from graphviz import Source

In [77]:
experiment = OpenmlClusteringExperiment(check_if_exists=False)
model_nickname = 'CoHiRF'
dataset_id = 61
seed_model = 4735
standardize = True
model_params = dict(components_size=29, repetitions=3, kmeans_n_clusters=5)
fit_params = dict()
results = experiment.run_openml_experiment_combination(model_nickname=model_nickname, seed_model=seed_model, dataset_id=dataset_id, standardize=standardize,
                                                       model_params=model_params, fit_params=fit_params, return_results=True, log_to_mlflow=True)

Running...
model_nickname: RecursiveClustering
seed_model: 4735
dataset_id: 61
model_params: {'components_size': 29, 'repetitions': 3, 'kmeans_n_clusters': 5}
fit_params: {}
standardize: True

Finished!
total_elapsed_time: 0.0878590370011807
model_nickname: RecursiveClustering
seed_model: 4735
dataset_id: 61
model_params: {'components_size': 29, 'repetitions': 3, 'kmeans_n_clusters': 5}
fit_params: {}
standardize: True



In [509]:
# first lets transform label_sequence in a graph, with nodes being clusters and edges being transitions between clusters (across iterations)
# inside each cluster we will store the samples that belong to that cluster
# we will first build the graph with dictionaries and then convert it to a graphviz Digraph object
label_sequence = results['load_model_return']['model'].labels_sequence_
y = results['load_data_return']['y']
max_samples = 5
n_iterations = label_sequence.shape[1]
n_samples = label_sequence.shape[0]
graph = dict()
max_samples_shown = 5
color_sequence = plt.cm.tab20.colors
y_codes = y.astype('category').cat.codes
y_colors = y_codes.map(lambda x: rgb2hex(color_sequence[x]))
for i in range(n_iterations):
    unique_clusters = np.unique(label_sequence[:, i])
    for cluster in unique_clusters:
        cluster_label = f"cluster_{cluster}_iter_{i}"
        samples_in_cluster_idx = np.where(label_sequence[:, i] == cluster)[0]
        samples_in_cluster = y[samples_in_cluster_idx]
        
        graph[cluster_label] = dict()
        graph[cluster_label]['samples'] = samples_in_cluster
        graph[cluster_label]['cluster'] = cluster
        graph[cluster_label]['iter'] = i
        
        if i > 0:
            graph[cluster_label]['prev_cluster'] = []
            previous_labels = label_sequence[:, i - 1]
            previous_clusters = np.unique(previous_labels[samples_in_cluster_idx])
            for prev_cluster in previous_clusters:
                prev_cluster_label = f"cluster_{prev_cluster}_iter_{i-1}"
                graph[cluster_label]['prev_cluster'].append(prev_cluster_label)

In [592]:
# now we will convert the graph to a graphviz in the dot language
dot_str = 'digraph G {\n'
# LR -> left to right
dot_str += "rankdir=LR;\n"
# compound must be true to allow subgraphs
dot_str += "compound=true;\n"

for cluster_label, cluster_dict in graph.items():
    label = cluster_dict['samples'].value_counts().sort_index()
    label.index.name = 'Class count'
    label = label.to_string()
    dot_str += f"{cluster_label} [label=\"{label}\", fontsize=15];\n"

for cluster_label, cluster_dict in graph.items():
    if 'prev_cluster' in cluster_dict:
        for prev_cluster in cluster_dict['prev_cluster']:
            dot_str += f"{prev_cluster} -> {cluster_label};\n"
            
dot_str += "}"

In [594]:
# render the graph
s = Source(dot_str, filename="iris_agg", format="pdf")
s.save()
s.view()

'iris_agg.pdf'